# Analyze the same bank cohort with Polars and DuckDB

A practical lazy-query and SQL lesson over a frozen 2026Q2 FDIC institution dataset. Use the same selected population and reporting date in both tools, compare results, and keep the source evidence with the exported Parquet. After installing the optional libraries, no source requests or account are needed.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

try:
    installed = importlib.metadata.version("us-bank-data") == "0.2.0"
except importlib.metadata.PackageNotFoundError:
    installed = False
if not installed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "us-bank-data==0.2.0"])

if any(importlib.util.find_spec(name) is None for name in ["polars", "duckdb", "pyarrow"]):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "us-bank-data[teaching]==0.2.0"])


In [ ]:
from pathlib import Path
from usbankdata import snapshot

data = snapshot().select(bank_classes=["N", "SM", "NM", "SB", "SI", "SL"],
                         min_assets_usd=100_000_000, max_assets_usd=3_000_000_000)
output = Path("outputs/polars-duckdb")
path = data.export(output / "bank-cohort.parquet")
assert data.rows
print("Institutions:", len(data.certs))


## Join identities and group with Polars

Read the Parquet lazily and join the source identity table by CERT, with a many-to-one validation. Group by the same report date and source state, aggregate accounting amounts in dollars, and count missing CET1 observations. A null-safe derived funding percentage is separate from the reported fields. Identity labels are as retrieved. Do not average bank ratios to represent an aggregate balance sheet. No ranking or sale availability is inferred.


In [ ]:
import polars as pl

identities = [{"cert": bank["cert"], "state": bank["state"]} for bank in data.banks]
identity_frame = pl.DataFrame(identities)
frame = (pl.scan_parquet(path)
    .join(identity_frame.lazy(), on="cert", how="left", validate="m:1")
    .with_columns(pl.when(pl.col("domestic_deposits_usd") > 0)
        .then(100 * pl.col("uninsured_deposits_usd") / pl.col("domestic_deposits_usd"))
        .otherwise(None).alias("uninsured_share_pct_derived")))
polars_result = (frame.group_by(["report_date", "state"]).agg(
    pl.len().alias("institutions"),
    pl.col("assets_usd").sum().alias("assets_usd"),
    pl.col("cet1_ratio_pct").null_count().alias("missing_cet1")
).sort(["report_date", "state"]).collect())
print(polars_result)
assert polars_result["institutions"].sum() == len(data.certs)
print(frame.select("cert", "uninsured_share_pct_derived").head(5).collect())


## Query the same file with DuckDB

The query uses a parameter for the file path. Read-only SQL keeps nulls intact. Compare counts, total assets and missing CET1 counts with Polars on exactly the same cohort.


In [ ]:
import duckdb
import pyarrow as pa

connection = duckdb.connect()
connection.register("identities", pa.Table.from_pylist(identities))
sql_result = connection.execute("""
    SELECT report_date, state, count(*) AS institutions, sum(assets_usd) AS assets_usd,
           count(*) - count(cet1_ratio_pct) AS missing_cet1
    FROM read_parquet(?) AS financials
    LEFT JOIN identities USING (cert)
    GROUP BY report_date, state
    ORDER BY report_date, state
""", [str(path)]).fetchall()
connection.close()
assert sql_result == polars_result.rows()
print(sql_result[:5])


## Verify the evidence embedded in Parquet

Raw source responses, hashes, metric definitions, coverage and selection rules are stored under `usbankdata` in the Parquet schema metadata. Keep them when creating a new file; a generic dataframe rewrite can discard that metadata.


In [ ]:
import json
import pyarrow.parquet as pq

receipt = json.loads(pq.read_schema(path).metadata[b"usbankdata"])
assert receipt["dataset_id"] == data.dataset_id
assert receipt["selection"]["anchor_report_date"] == "2026-06-30"
assert receipt["metric_definitions"]["assets_usd"]["unit"] == "USD"
assert receipt["metric_definitions"]["cet1_ratio_pct"]["unit"] == "percent"
print(receipt["coverage"])


## Exercise

Compare deposits and assets for another explicit size range. Add a safe derived loans/assets measure while preserving missing fields and zero denominators. Ensure Polars and SQL use the same selection. Source: FDIC BankFind Suite; BanksOnSale compilation and metadata (CC BY 4.0), provided through US Bank Data. This is an as-retrieved dataset edition, not an original-filing vintage. [Population and units](https://github.com/anshuman-pandey/us-bank-data/blob/v0.2.0/docs/COHORTS.md).
